# README
- **Author**: 詹明翰、謝子詮、曾翊涵（沿用已確認共同作者，不推定個別分工）
- **Created At**: 2026-11-26
- **Last Modified At**: 2026-11-26

## What does this file do?
Unit 1.3驗證研究24個月人口資料、官方代碼與歷史快照，建立唯一正式衍生資料`village_crosswalk.csv`，並於記憶體驗證Dataset B1彙整。
第一階段精確區里名稱配對；第二階段僅在455個已配對里驗證區碼對照＋里碼後三碼規律後，將糖廍里列為高可信但未官方跨碼認證的候選。
不模糊配對、不修改原始名稱或geometry、不輸出年度人口資料、不執行事故歸里。

## What does this file take?
六個原始資料hash綁定Unit 0.5；其餘為已執行來源說明／里界驗證及新增官方證據包。
- `data/raw/112迄今各里人口數按年齡分.csv`；SHA-256 `a663a4f0329b18bb0514da7fd9f6845312d08fadef71e566faf1930c03541641`
- `data/raw/G97_A_CAVLGE_P.shp`；SHA-256 `f7fb4813a2283198ca564f5686823c454c5fd832d2f6981a2ee88a94719be3cc`
- `data/raw/G97_A_CAVLGE_P.shx`；SHA-256 `ab7bc5e63229f676d3580b68e98db59b91a2d327c74ca23b0000ca1c568d414e`
- `data/raw/G97_A_CAVLGE_P.dbf`；SHA-256 `2f43f92e84279a45b2512cf453e9b72f706d1ad1402665c946b1a354f75d8577`
- `data/raw/G97_A_CAVLGE_P.prj`；SHA-256 `8dac7d82ea889063f9eaa7df408fcaeb7307d32831ea973b6534d0139a4523c5`
- `data/raw/G97_A_CAVLGE_P.cpg`；SHA-256 `3ad3031f5503a4404af825262ee8232cc04d4ea6683d42c5dd0a2f2a27ac9824`
- `data/raw/taipei_population_age_raw_documentation.ipynb`；SHA-256 `8b7b5376b6e79821c0cdaffd7f17e97f9e466bb3933d6b77552cf48846c29f53`
- `data/raw/taipei_village_boundaries_raw_documentation.ipynb`；SHA-256 `74c1c7e8fca3f5c5a82a94282f25dd42b6c2ba91e952e6408c3bd307fb269fcf`
- `data/code/taipei_village_boundary_validation.ipynb`；SHA-256 `662dbee2dc42c3552b12628a1e89fcc81b07155c225277117f50d07e0145a343`
- `data/raw/unit13_official_code_evidence.zip`；SHA-256 `9f27bd2f9211f4c21253876529efec1414b82f0b7014f6d4028b70f66b58cc40`
- `data/raw/unit13_official_code_evidence_documentation.ipynb`；SHA-256 `08f9972f534920668134f919c6f460c915918e92bd599031dcec61c006589682`

人口[原始說明](../raw/taipei_population_age_raw_documentation.ipynb)、里界[原始說明](../raw/taipei_village_boundaries_raw_documentation.ipynb)、[Unit 1.2驗證](taipei_village_boundary_validation.ipynb)、[官方代碼證據說明](../raw/unit13_official_code_evidence_documentation.ipynb)。
[PROJECT.md](../../PROJECT.md)為研究脈絡，不因其獲准更新而重設任何資料hash。

## What does this file output?
- Data path / output key: `data/processed/village_crosswalk.csv` / `village_crosswalk`。
- Role: `crosswalk`；每列一個固定研究里，11欄、456列，三個識別欄均唯一字串，含驗證狀態與來源證據。
- Documentation: [對照表說明](../processed/village_crosswalk_documentation.ipynb)。
- Verification status / SHA-256: 以最後重新讀檔及hash實際輸出為準；驗證檔案與記憶體內容一致後才交接。
- Downstream input: `data/processed/village_crosswalk.csv`；正式整合必須明確檢查`verification_status`，不得默認暫定配對已通過。
- 年度912列及兩年456列僅在記憶體試算；必要核對摘要存Git忽略的`data/temp/unit13_validation/`，不建立年度分析資料檔。


In [1]:
from pathlib import Path
import hashlib,json,sys,platform,os,io,zipfile,re
import numpy as np
import pandas as pd
from IPython.display import display,Markdown
def project_root():
    for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents):
        if (p/'AGENTS.md').is_file() and (p/'templates').is_dir():return p
    raise FileNotFoundError('Open within the student project containing AGENTS.md and templates/.')
ROOT=project_root()
assert Path(sys.executable).resolve()==(ROOT/'.venv/Scripts/python.exe').resolve()
assert sys.prefix!=sys.base_prefix
os.environ['PROJ_NETWORK']='OFF'
def file(relative,area='data',must_exist=True):
    rel=Path(relative)
    if rel.is_absolute() or '..' in rel.parts:raise ValueError('Explicit project-relative path required')
    path=(ROOT/rel).resolve()
    if not path.is_relative_to((ROOT/area).resolve()):raise ValueError('Path outside intended project area')
    if must_exist and not path.is_file():raise FileNotFoundError(relative)
    return path
def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as stream:
        for b in iter(lambda:stream.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
def verify(expected):
    rows=[]
    for relative,digest in expected.items():
        path=file(relative)
        assert sha(path)==digest,f'Input changed; stop and inspect version: {relative}'
        rows.append({'path':relative,'bytes':path.stat().st_size,'sha256':digest})
    return pd.DataFrame(rows)
def show(title,table,block=20):
    display(Markdown('**'+title+'**'))
    if not len(table):display(Markdown('無符合紀錄。'))
    for start in range(0,len(table),block):display(table.iloc[start:start+block])
pd.set_option('display.max_colwidth',110)
print({'python':platform.python_version(),'interpreter_is_project_venv':True,'network_in_notebook':'none'})

EXPECTED_SHA256={'data/raw/112迄今各里人口數按年齡分.csv': 'a663a4f0329b18bb0514da7fd9f6845312d08fadef71e566faf1930c03541641', 'data/raw/G97_A_CAVLGE_P.shp': 'f7fb4813a2283198ca564f5686823c454c5fd832d2f6981a2ee88a94719be3cc', 'data/raw/G97_A_CAVLGE_P.shx': 'ab7bc5e63229f676d3580b68e98db59b91a2d327c74ca23b0000ca1c568d414e', 'data/raw/G97_A_CAVLGE_P.dbf': '2f43f92e84279a45b2512cf453e9b72f706d1ad1402665c946b1a354f75d8577', 'data/raw/G97_A_CAVLGE_P.prj': '8dac7d82ea889063f9eaa7df408fcaeb7307d32831ea973b6534d0139a4523c5', 'data/raw/G97_A_CAVLGE_P.cpg': '3ad3031f5503a4404af825262ee8232cc04d4ea6683d42c5dd0a2f2a27ac9824', 'data/raw/taipei_population_age_raw_documentation.ipynb': '8b7b5376b6e79821c0cdaffd7f17e97f9e466bb3933d6b77552cf48846c29f53', 'data/raw/taipei_village_boundaries_raw_documentation.ipynb': '74c1c7e8fca3f5c5a82a94282f25dd42b6c2ba91e952e6408c3bd307fb269fcf', 'data/code/taipei_village_boundary_validation.ipynb': '662dbee2dc42c3552b12628a1e89fcc81b07155c225277117f50d07e0145a343', 'data/raw/unit13_official_code_evidence.zip': '9f27bd2f9211f4c21253876529efec1414b82f0b7014f6d4028b70f66b58cc40', 'data/raw/unit13_official_code_evidence_documentation.ipynb': '08f9972f534920668134f919c6f460c915918e92bd599031dcec61c006589682'}
show('表1：原始與銜接輸入逐檔hash',verify(EXPECTED_SHA256),block=6)
import geopandas as gpd
population=pd.read_csv(file('data/raw/112迄今各里人口數按年齡分.csv'),encoding='cp950',dtype='string',keep_default_na=False)
boundary=gpd.read_file(file('data/raw/G97_A_CAVLGE_P.shp'),encoding='utf-8',engine='pyogrio')
original_geometry=boundary.geometry.to_wkb().copy()
assert boundary.crs.to_epsg()==3826 and len(boundary)==456
assert boundary.CPTVID.map(lambda value:isinstance(value,str)).all()
print({'population_rows':len(population),'population_columns':len(population.columns),
       'geometry_read_only':True,'source_hashes_match_Unit_0_5':True,'existing_notebooks_executed':False})


{'python': '3.12.2', 'interpreter_is_project_venv': True, 'network_in_notebook': 'none'}


**表1：原始與銜接輸入逐檔hash**

,path,bytes,sha256
0,data/raw/112迄今各里人口數按年齡分.csv,21284860,a663a4f0329b18bb0514da7fd9f6845312d08fadef71e566faf1930c03541641
1,data/raw/G97_A_CAVLGE_P.shp,765784,f7fb4813a2283198ca564f5686823c454c5fd832d2f6981a2ee88a94719be3cc
2,data/raw/G97_A_CAVLGE_P.shx,3748,ab7bc5e63229f676d3580b68e98db59b91a2d327c74ca23b0000ca1c568d414e
3,data/raw/G97_A_CAVLGE_P.dbf,272706,2f43f92e84279a45b2512cf453e9b72f706d1ad1402665c946b1a354f75d8577
4,data/raw/G97_A_CAVLGE_P.prj,399,8dac7d82ea889063f9eaa7df408fcaeb7307d32831ea973b6534d0139a4523c5
5,data/raw/G97_A_CAVLGE_P.cpg,5,3ad3031f5503a4404af825262ee8232cc04d4ea6683d42c5dd0a2f2a27ac9824


,path,bytes,sha256
6,data/raw/taipei_population_age_raw_documentation.ipynb,284126,8b7b5376b6e79821c0cdaffd7f17e97f9e466bb3933d6b77552cf48846c29f53
7,data/raw/taipei_village_boundaries_raw_documentation.ipynb,106813,74c1c7e8fca3f5c5a82a94282f25dd42b6c2ba91e952e6408c3bd307fb269fcf
8,data/code/taipei_village_boundary_validation.ipynb,1802468,662dbee2dc42c3552b12628a1e89fcc81b07155c225277117f50d07e0145a343
9,data/raw/unit13_official_code_evidence.zip,300990,9f27bd2f9211f4c21253876529efec1414b82f0b7014f6d4028b70f66b58cc40
10,data/raw/unit13_official_code_evidence_documentation.ipynb,55667,08f9972f534920668134f919c6f460c915918e92bd599031dcec61c006589682


{'population_rows': 64722, 'population_columns': 107, 'geometry_read_only': True, 'source_hashes_match_Unit_0_5': True, 'existing_notebooks_executed': False}


## 1. Research-period Population Validation
原始CSV以CP950嚴格解碼、代碼全字串；先區分8碼市／區彙總與11碼里，再選民國113、114年。
本單元核對全部研究里列的總計與101年齡欄；只在記憶體解析數值，缺失／非數值／負值不補零。
數量不符時先保存診斷並停止，不藉刪重或補月份強迫符合預期。


In [2]:
AGE=[f'{age}歲數量' for age in range(100)]+['100歲以上']
NUMERIC=['總計']+AGE
assert set(AGE).issubset(population.columns) and len(AGE)==101
scope=population.loc[population['年份'].isin(['113','114'])].copy()
show('表2：研究期間各代碼長度的原始列數',scope['區域代碼'].str.len().value_counts().rename('rows').to_frame())
research=scope.loc[scope['區域代碼'].str.len().eq(11)].copy()
numeric=research[NUMERIC].apply(pd.to_numeric,errors='coerce')
bad_missing=research[NUMERIC].eq('')
bad_numeric=numeric.isna() & ~bad_missing
bad_negative=numeric.lt(0)
numeric_diagnostic={'research_village_rows':len(research),'age_columns':len(AGE),
   'missing_population_cells':int(bad_missing.to_numpy().sum()),'nonnumeric_cells':int(bad_numeric.to_numpy().sum()),
   'negative_population_cells':int(bad_negative.to_numpy().sum()),
   'nonfinite_population_cells':int((~np.isfinite(numeric.to_numpy(dtype=float))).sum()),
   'noninteger_population_cells':int((numeric.mod(1).ne(0)).to_numpy().sum()),
   'duplicate_year_month_code_sex':int(research.duplicated(['年份','月份','區域代碼','性別']).sum()),
   'age_total_mismatch_rows':int(numeric[AGE].sum(axis=1).ne(numeric['總計']).sum())}
print(numeric_diagnostic)
show('表3：非數值／缺失／負值異常原始列',research.loc[(bad_missing|bad_numeric|bad_negative).any(axis=1),['年份','月份','區域代碼','區域別','性別']])
assert not (bad_missing|bad_numeric|bad_negative).to_numpy().any()
assert numeric_diagnostic['nonfinite_population_cells']==numeric_diagnostic['noninteger_population_cells']==0
assert numeric_diagnostic['duplicate_year_month_code_sex']==numeric_diagnostic['age_total_mismatch_rows']==0
research=pd.concat([research.drop(columns=NUMERIC),numeric.astype('int64')],axis=1).copy()
research['year']=pd.to_numeric(research['年份']).astype(int)+1911
research['month']=pd.to_numeric(research['月份']).astype(int)
month_table=research.groupby(['year','month']).agg(rows=('區域代碼','size'),villages=('區域代碼','nunique'),sex_categories=('性別','nunique'))
month_table['total_sex_rows']=research.loc[research['性別'].eq('計')].groupby(['year','month']).size()
show('表4：24個月完整性',month_table,block=24)
groups=research.groupby(['year','month','區域代碼'])['性別'].agg(lambda values:tuple(sorted(values)))
gender_wide=research.pivot(index=['year','month','區域代碼'],columns='性別',values=NUMERIC)
gender_diff=gender_wide.xs('男',level='性別',axis=1)+gender_wide.xs('女',level='性別',axis=1)-gender_wide.xs('計',level='性別',axis=1)
gender_mismatch=int(gender_diff.ne(0).to_numpy().sum())
code_sets={year:set(research.loc[research.year.eq(year),'區域代碼']) for year in [2024,2025]}
name_changes=research.groupby('區域代碼')['區域別'].nunique().rename('observed_name_count')
show('表5：研究期同代碼有多個原始名稱者',name_changes.loc[name_changes.gt(1)].to_frame())
print({'male_plus_female_mismatch_cells':gender_mismatch,'sex_set_mismatch_groups':int(groups.map(lambda value:value!=tuple(sorted(['計','男','女']))).sum()),
       '2024_2025_code_sets_equal':code_sets[2024]==code_sets[2025],'observed_name_change_codes':int(name_changes.gt(1).sum()),
       'total_sex_rows':int(research['性別'].eq('計').sum()),'theoretical_annual_rows':len(code_sets[2024])*2})
assert set(month_table.index)=={(y,m) for y in [2024,2025] for m in range(1,13)}
assert month_table['rows'].eq(1368).all() and month_table.villages.eq(456).all()
assert month_table.total_sex_rows.eq(456).all()
assert groups.map(lambda value:value==tuple(sorted(['計','男','女']))).all() and gender_mismatch==0
assert code_sets[2024]==code_sets[2025] and len(research)==32832
assert not name_changes.gt(1).any()


**表2：研究期間各代碼長度的原始列數**

,rows
區域代碼,
11,32832
8,936


{'research_village_rows': 32832, 'age_columns': 101, 'missing_population_cells': 0, 'nonnumeric_cells': 0, 'negative_population_cells': 0, 'nonfinite_population_cells': 0, 'noninteger_population_cells': 0, 'duplicate_year_month_code_sex': 0, 'age_total_mismatch_rows': 0}


**表3：非數值／缺失／負值異常原始列**

無符合紀錄。

**表4：24個月完整性**

rows  villages  sex_categories  total_sex_rows
year month                                                
2024 1      1368       456               3             456
     2      1368       456               3             456
     3      1368       456               3             456
     4      1368       456               3             456
     5      1368       456               3             456
     6      1368       456               3             456
     7      1368       456               3             456
     8      1368       456               3             456
     9      1368       456               3             456
     10     1368       456               3             456
     11     1368       456               3             456
     12     1368       456               3             456
2025 1      1368       456               3             456
     2      1368       456               3             456
     3      1368       456               3             456
     4      1368       456               3             456
     5      1368       456               3             456
     6      1368       456               3             456
     7      1368       456               3             456
     8      1368       456               3             456
     9      1368       456               3             456
     10     1368       456               3             456
     11     1368       456               3             456
     12     1368       456               3             456

**表5：研究期同代碼有多個原始名稱者**

無符合紀錄。

{'male_plus_female_mismatch_cells': 0, 'sex_set_mismatch_groups': 0, '2024_2025_code_sets_equal': True, 'observed_name_change_codes': 0, 'total_sex_rows': 10944, 'theoretical_annual_rows': 912}


<a id="sections-2-3"></a>

## 2. Official Code Evidence and History
戶政司168頁將RSCD0103稱為「省市縣市鄉鎮市區代碼」、型式X(08)；VillageUTF8列明村里碼、所屬8碼區碼及名稱。
11碼＝8碼所屬區碼＋3碼尾碼在官方各快照及本地人口里列實際核對，不將這個關係擴張為里界CPTVID的官方轉換規格。
10碼CPTVID的正式編碼文件仍未取得；本機觀察為7碼CPTID＋3碼里尾碼，CPID為5碼，層級關係逐列核對。
來源網址／版本／原始bytes與hash完整保存在[來源包說明](../raw/unit13_official_code_evidence_documentation.ipynb)；本Notebook不連網。
11109是研究前快照，11406／09／11為2025年快照，11509為較後快照；不是2024／2025逐月官方異動清冊，穩定名稱／代碼不證明幾何未動。


In [3]:
archive=zipfile.ZipFile(file('data/raw/unit13_official_code_evidence.zip'))
source_manifest=json.loads(archive.read('source_manifest.json').decode('utf-8'))
for item in source_manifest:assert hashlib.sha256(archive.read(item['member'])).hexdigest()==item['sha256']
show('表6：官方來源、版本與hash',pd.DataFrame(source_manifest)[['member','source_url','version','sha256']],block=5)
TABLE_NAMES=['11109VillageUTF8.txt','11406VillageUTF8.txt','11409VillageUTF8.txt','11411VillageUTF8.txt','11509VillageUTF8.txt']
official={name:pd.read_csv(io.BytesIO(archive.read(name)),header=None,dtype='string',encoding='utf-8-sig',
         names=['population_code','population_district_code','official_name'],keep_default_na=False) for name in TABLE_NAMES}
district_map={line.split('=',1)[0]:line.split('=',1)[1] for line in archive.read('RSCD0103.txt').decode('cp950').splitlines() if '=' in line}
districts=scope.loc[scope['區域代碼'].str.len().eq(8)&scope['區域別'].ne('總計'),['區域代碼','區域別']].drop_duplicates()
assert districts['區域代碼'].is_unique and districts['區域別'].is_unique and len(districts)==12
assert all(district_map[code]=='臺北市'+name for code,name in districts.itertuples(index=False,name=None))
district_names=dict(districts.itertuples(index=False,name=None))
names=research[['區域代碼','區域別']].drop_duplicates().rename(columns={'區域代碼':'population_code','區域別':'population_village_name'})
assert names.population_code.is_unique and len(names)==456
names['population_district_code']=names.population_code.str[:8]
names['district_name']=names.population_district_code.map(district_names)
assert names.district_name.notna().all()
temporal=[];taipei_tables={}
for name,table in official.items():
    assert table.population_code.is_unique and table.population_code.str[:8].eq(table.population_district_code).all()
    taipei=table.loc[table.population_code.str.startswith('63000')].copy()
    taipei_tables[name]=taipei.set_index('population_code').sort_index()
    joined=names.merge(taipei,on=['population_code','population_district_code'],how='left',validate='one_to_one',indicator=True)
    temporal.append({'version':name,'taipei_villages':len(taipei),'same_codes_as_research':set(taipei.population_code)==code_sets[2024],
                     'code_not_in_official_table':int(joined['_merge'].ne('both').sum()),
                     'raw_name_differs_from_official':int(joined.population_village_name.ne(joined.official_name).sum())})
    assert set(taipei.population_code)==code_sets[2024]
show('表7：全部456個研究里與五版本官方碼表比較',pd.DataFrame(temporal))
reference=taipei_tables[TABLE_NAMES[0]]
assert all(taipei_tables[name].equals(reference) for name in TABLE_NAMES)
current=taipei_tables['11509VillageUTF8.txt']
names['official_village_name']=names.population_code.map(current.official_name)
show('表8：人口原始名稱與官方名稱不一致者',names.loc[names.population_village_name.ne(names.official_village_name)])
print({'official_taipei_code_and_name_rows_identical_across_five_snapshots':True,
       'official_village_prefix_matches_8_digit_parent':True,'official_2024_2025_geometry_equality':'not established',
       'boundary_10_digit_code_specification':'not obtained; empirical relationship only'})


**表6：官方來源、版本與hash**

,member,source_url,version,sha256
0,ris_168.html,https://www.ris.gov.tw/documents/html/5/1/168.html,retrieved 2026-11-26; code catalog,ad1152ddbb6d5c01c41fffb28f079dae3566bc72044ee529611d6e054fd59938
1,ris_167.html,https://www.ris.gov.tw/documents/html/5/1/167.html,retrieved 2026-11-26; current code index,bf879d3314c871ec4260677802f9d3bd6a5c827f7b78a0a283bdd58e3e244c63
2,ris_558.html,https://www.ris.gov.tw/documents/html/5/1/558.html,retrieved 2026-11-26; historical code index,1fb066026a51789a3e1c36738565d67718060f005bf333a3ab1d6f2f0cd7d7e6
3,RSCD0103.txt,https://www.ris.gov.tw/documents/data/5/1/RSCD0103.txt,"current; official town code table, no effective-date assertion",c89a4bfc564aa3fa5c9c59fd0656a52c8b092c62b812f1348ac8342f7b4945e6
4,11109VillageUTF8.txt,https://www.ris.gov.tw/documents/data/5/1/11109VillageUTF8.txt,11109 / 2022-09,78a53459ac359e2e1c79e29ea32751b5fd46e8241fa068dcd7f6fd41108e9242


,member,source_url,version,sha256
5,11406VillageUTF8.txt,https://www.ris.gov.tw/documents/data/5/1/11406VillageUTF8.txt,11406 / 2025-06,99405e9a3e62004d341cdd17d4a640eed49be1deddf48d5b92c060469b4ae0e4
6,11409VillageUTF8.txt,https://www.ris.gov.tw/documents/data/5/1/11409VillageUTF8.txt,11409 / 2025-09,fc898ed6a6570adc4110cd54a6e75233a5996332dda9735a2b8889982d3d9b12
7,11411VillageUTF8.txt,https://www.ris.gov.tw/documents/data/5/1/11411VillageUTF8.txt,11411 / 2025-11,740c39e4c21e7fcc8fddf9741eee26c85724558f8cee0c1f357e67ee252ed484
8,11509VillageUTF8.txt,https://www.ris.gov.tw/documents/data/5/1/11509VillageUTF8.txt,11509 / 2026-09,c7241fd8f93bc7d330e4fc52b89365a8fa7a8f695bb09012dce01c5dfbfa24b6


**表7：全部456個研究里與五版本官方碼表比較**

,version,taipei_villages,same_codes_as_research,code_not_in_official_table,raw_name_differs_from_official
0,11109VillageUTF8.txt,456,True,0,1
1,11406VillageUTF8.txt,456,True,0,1
2,11409VillageUTF8.txt,456,True,0,1
3,11411VillageUTF8.txt,456,True,0,1
4,11509VillageUTF8.txt,456,True,0,1


**表8：人口原始名稱與官方名稱不一致者**

,population_code,population_village_name,population_district_code,district_name,official_village_name
17625,63000070015,糖?里,63000070,萬華區,糖廍里


{'official_taipei_code_and_name_rows_identical_across_five_snapshots': True, 'official_village_prefix_matches_8_digit_parent': True, 'official_2024_2025_geometry_equality': 'not established', 'boundary_10_digit_code_specification': 'not obtained; empirical relationship only'}


## 3. Exact Matching, Local Code Relationship and Tangbu Exception
只用行政區＋原始里名做精確名稱配對，先驗證兩端唯一；不同區同名不能以里名單獨合併。
區碼對照以12區名稱及戶政司區碼表核對，再於455個精確配對里驗證「對應CPTID＋人口里碼末三碼」。
這是此批來源的可驗證規律，不是全臺或歷史通用轉換規格；糖廍里不因這個規律升級為官方跨碼認證。


In [4]:
bnames=boundary[['CPTVID','CPTID','CPID','TNAME','VNAME']].rename(columns={'CPTVID':'boundary_code','CPTID':'boundary_district_code',
         'TNAME':'district_name','VNAME':'boundary_village_name'})
assert bnames.boundary_code.is_unique and not bnames.boundary_code.isna().any()
assert not names.duplicated(['district_name','population_village_name']).any()
assert not bnames.duplicated(['district_name','boundary_village_name']).any()
exact=names.merge(bnames,left_on=['district_name','population_village_name'],right_on=['district_name','boundary_village_name'],
                  how='left',validate='one_to_one',indicator=True)
show('表9：原始精確名稱未配對的人口里',exact.loc[exact['_merge'].ne('both'),['population_code','district_name','population_village_name','official_village_name']])
assert exact['_merge'].eq('both').sum()==455
show('表10：全市跨行政區同名里；不能只以里名合併',bnames.loc[bnames.boundary_village_name.duplicated(keep=False),['district_name','boundary_village_name','boundary_code']])
local_district=bnames[['district_name','boundary_district_code','CPID']].drop_duplicates()
assert len(local_district)==12 and local_district.boundary_district_code.is_unique
district_bridge=districts.rename(columns={'區域代碼':'population_district_code','區域別':'district_name'}).merge(local_district,on='district_name',validate='one_to_one')
show('表11：12區的官方人口區碼與本機里界區碼對照',district_bridge)
parent_bridge=dict(zip(district_bridge.population_district_code,district_bridge.boundary_district_code))
names['candidate_boundary_code']=names.population_district_code.map(parent_bridge)+names.population_code.str[-3:]
assert bnames.boundary_code.str[:7].eq(bnames.boundary_district_code).all()
assert bnames.boundary_district_code.str[:5].eq(bnames.CPID).all()
exact_checked=exact.loc[exact['_merge'].eq('both')].copy()
predicted=exact_checked.population_district_code.map(parent_bridge)+exact_checked.population_code.str[-3:]
assert predicted.eq(exact_checked.boundary_code).all() and predicted.is_unique
assert names.candidate_boundary_code.is_unique and set(names.candidate_boundary_code)==set(bnames.boundary_code)
proposed=names.merge(bnames,left_on='candidate_boundary_code',right_on='boundary_code',how='left',validate='one_to_one',suffixes=('','_boundary'))
assert proposed.district_name.eq(proposed.district_name_boundary).all()
tangbu=proposed.loc[proposed.population_code.eq('63000070015')]
assert len(tangbu)==1 and tangbu.boundary_code.iloc[0]=='6300700015'
assert tangbu.population_village_name.iloc[0]=='糖?里' and tangbu.boundary_village_name.iloc[0]=='糖蔀里'
assert tangbu.official_village_name.iloc[0]=='糖廍里'
show('表12：糖廍里來源名稱及兩端代碼（原值保留）',tangbu[['population_code','boundary_code','district_name','population_village_name','boundary_village_name','official_village_name']])
show('表13：五個官方版本的糖廍里原列',pd.concat([t.loc[t.index=='63000070015'].reset_index().assign(version=name) for name,t in taipei_tables.items()],ignore_index=True))
print({'exact_district_village_name_matches':455,'empirical_rule_checks_passed':455,'proposed_pairs':456,
       'cross_district_mismatches':0,'Tangbu_population_code_and_official_name':'supported by five official snapshots',
       'Tangbu_official_boundary_code_bridge':'not obtained; keep provisional_match'})


**表9：原始精確名稱未配對的人口里**

,population_code,district_name,population_village_name,official_village_name
239,63000070015,萬華區,糖?里,糖廍里


**表10：全市跨行政區同名里；不能只以里名合併**

,district_name,boundary_village_name,boundary_code
68,北投區,中央里,6301200022
239,中山區,中央里,6300400033
265,士林區,新安里,6301100045
316,萬華區,新安里,6300700021


**表11：12區的官方人口區碼與本機里界區碼對照**

,population_district_code,district_name,boundary_district_code,CPID
0,63000010,松山區,6300100,63001
1,63000020,信義區,6300200,63002
2,63000030,大安區,6300300,63003
3,63000040,中山區,6300400,63004
4,63000050,中正區,6300500,63005
5,63000060,大同區,6300600,63006
6,63000070,萬華區,6300700,63007
7,63000080,文山區,6300800,63008
8,63000090,南港區,6300900,63009
9,63000100,內湖區,6301000,63010


**表12：糖廍里來源名稱及兩端代碼（原值保留）**

,population_code,boundary_code,district_name,population_village_name,boundary_village_name,official_village_name
239,63000070015,6300700015,萬華區,糖?里,糖蔀里,糖廍里


**表13：五個官方版本的糖廍里原列**

,population_code,population_district_code,official_name,version
0,63000070015,63000070,糖廍里,11109VillageUTF8.txt
1,63000070015,63000070,糖廍里,11406VillageUTF8.txt
2,63000070015,63000070,糖廍里,11409VillageUTF8.txt
3,63000070015,63000070,糖廍里,11411VillageUTF8.txt
4,63000070015,63000070,糖廍里,11509VillageUTF8.txt


{'exact_district_village_name_matches': 455, 'empirical_rule_checks_passed': 455, 'proposed_pairs': 456, 'cross_district_mismatches': 0, 'Tangbu_population_code_and_official_name': 'supported by five official snapshots', 'Tangbu_official_boundary_code_bridge': 'not obtained; keep provisional_match'}


## 4. Crosswalk Schema and Verification Status
`village_id`採唯一CPTVID，僅作本研究共同鍵，並非宣稱兩個機關編碼相同。
狀態區分：verified_official＝充分官方跨碼對照證據；verified_exact_match＝區里名與唯一性驗證；
provisional_match＝合理但未完證的候選；unmatched＝未配對。精確名稱配對不等於官方認證。
本批455里verified_exact_match、糖廍里1筆provisional_match；保留全456里，後續不得無聲排除或默認全部可用。


In [5]:
COLUMNS=['village_id','population_code','boundary_code','district_name','population_village_name','boundary_village_name',
         'standard_village_name','match_method','verification_status','source_evidence','notes']
crosswalk=pd.DataFrame({'village_id':proposed.boundary_code,'population_code':proposed.population_code,'boundary_code':proposed.boundary_code,
          'district_name':proposed.district_name,'population_village_name':proposed.population_village_name,
          'boundary_village_name':proposed.boundary_village_name,'standard_village_name':proposed.official_village_name})
equal_names=crosswalk.population_village_name.eq(crosswalk.boundary_village_name)
crosswalk['match_method']=np.where(equal_names,'district_and_village_exact','official_population_code_plus_empirical_local_rule')
crosswalk['verification_status']=np.where(equal_names,'verified_exact_match','provisional_match')
source_ref=('data/raw/112迄今各里人口數按年齡分.csv@'+EXPECTED_SHA256['data/raw/112迄今各里人口數按年齡分.csv']+
    ';data/raw/G97_A_CAVLGE_P.dbf@'+EXPECTED_SHA256['data/raw/G97_A_CAVLGE_P.dbf']+
    ';data/raw/unit13_official_code_evidence.zip@'+EXPECTED_SHA256['data/raw/unit13_official_code_evidence.zip']+
    '#RSCD0103.txt,11109VillageUTF8.txt,11406VillageUTF8.txt,11409VillageUTF8.txt,11411VillageUTF8.txt,11509VillageUTF8.txt'+
    ';https://www.ris.gov.tw/documents/html/5/1/558.html;data/code/taipei_population_validation.ipynb#sections-2-3')
crosswalk['source_evidence']=source_ref
crosswalk['notes']=np.where(equal_names,
    '行政區及原始里名精確一對一；官方人口名稱碼表旁證，不代表官方跨碼認證或歷史幾何相同。',
    '高可信候選：官方五版本均為糖廍里63000070015；本機糖蔀里6300700015符合455里規律，但缺官方跨碼／名稱誤植更正證明。正式整合須明確審查狀態。')
crosswalk=crosswalk[COLUMNS].astype('string').sort_values('village_id').reset_index(drop=True)
VERIFIED=['verified_official','verified_exact_match']
assert crosswalk[['village_id','population_code','boundary_code']].nunique().eq(456).all()
assert crosswalk.village_id.eq(crosswalk.boundary_code).all()
assert not crosswalk.isna().to_numpy().any() and not crosswalk.eq('').to_numpy().any()
assert crosswalk.verification_status.value_counts().to_dict()=={'verified_exact_match':455,'provisional_match':1}
for year in [2024,2025]:assert code_sets[year]==set(crosswalk.population_code)
show('表14：狀態分類；沒有以官方人口碼查證冒稱官方跨碼認證',crosswalk.verification_status.value_counts().reindex(
     ['verified_official','verified_exact_match','provisional_match','unmatched'],fill_value=0).rename('villages').to_frame())
show('表15：crosswalk前十列（識別與配對欄）',crosswalk.iloc[:10,:9])
show('表16：暫定／未配對紀錄',crosswalk.loc[~crosswalk.verification_status.isin(VERIFIED)].drop(columns='source_evidence'))
show('表17：各行政區配對狀態',pd.crosstab(crosswalk.district_name,crosswalk.verification_status))


**表14：狀態分類；沒有以官方人口碼查證冒稱官方跨碼認證**

,villages
verification_status,
verified_official,0
verified_exact_match,455
provisional_match,1
unmatched,0


**表15：crosswalk前十列（識別與配對欄）**

,village_id,population_code,boundary_code,district_name,population_village_name,boundary_village_name,standard_village_name,match_method,verification_status
0,6300100002,63000010002,6300100002,松山區,莊敬里,莊敬里,莊敬里,district_and_village_exact,verified_exact_match
1,6300100003,63000010003,6300100003,松山區,東榮里,東榮里,東榮里,district_and_village_exact,verified_exact_match
2,6300100004,63000010004,6300100004,松山區,三民里,三民里,三民里,district_and_village_exact,verified_exact_match
3,6300100005,63000010005,6300100005,松山區,新益里,新益里,新益里,district_and_village_exact,verified_exact_match
4,6300100006,63000010006,6300100006,松山區,富錦里,富錦里,富錦里,district_and_village_exact,verified_exact_match
5,6300100007,63000010007,6300100007,松山區,新東里,新東里,新東里,district_and_village_exact,verified_exact_match
6,6300100008,63000010008,6300100008,松山區,富泰里,富泰里,富泰里,district_and_village_exact,verified_exact_match
7,6300100009,63000010009,6300100009,松山區,介壽里,介壽里,介壽里,district_and_village_exact,verified_exact_match
8,6300100010,63000010010,6300100010,松山區,精忠里,精忠里,精忠里,district_and_village_exact,verified_exact_match
9,6300100011,63000010011,6300100011,松山區,東光里,東光里,東光里,district_and_village_exact,verified_exact_match


**表16：暫定／未配對紀錄**

,village_id,population_code,boundary_code,district_name,population_village_name,boundary_village_name,standard_village_name,match_method,verification_status,notes
239,6300700015,63000070015,6300700015,萬華區,糖?里,糖蔀里,糖廍里,official_population_code_plus_empirical_local_rule,provisional_match,高可信候選：官方五版本均為糖廍里63000070015；本機糖蔀里6300700015符合455里規律，但缺官方跨碼／名稱誤植更正證明。正式整合須明確審查狀態。


**表17：各行政區配對狀態**

verification_status,provisional_match,verified_exact_match
district_name,,
中山區,0,42
中正區,0,31
信義區,0,41
內湖區,0,39
北投區,0,42
南港區,0,20
士林區,0,51
大同區,0,25
大安區,0,53


## 5. Dataset B1 — Confirmed Main Feature Definitions
使用性別「計」；總人口`總計`，兒童為`0歲數量`至`14歲數量`，高齡為`65歲數量`至`99歲數量`加`100歲以上`。
年度需12個完整月份，兩年度合併需24個完整月份；缺月／缺人口不補零、不以可得月份冒充全年。

| 變數／中文 | 分子與分母、單位 | 層級、用途及限制 |
| --- | --- | --- |
| `population_density` 設籍人口密度 | 年度：Σ月總人口÷12，再÷EPSG:3826原geometry面積km²；合併：Σ24月總人口÷24，再÷同一面積；人/km² | 里×年或里×2024–2025合併；Dataset B1環境候選。固定原圖資、不用AREA；面積非正／缺失或配對未通過則主要密度NA，候選配對密度僅另作診斷。 |
| `elderly_population_ratio` 65歲以上人口比例 | Σ月份65歲以上人口÷Σ同期間總人口；0–1比例 | 同一時間／空間層級；不是年度比例的簡單平均。分母零或人口缺失／缺月則NA；不將人口結構解釋為個人事故機率。 |
| `child_population_ratio` 0–14歲人口比例 | Σ月份0–14歲人口÷Σ同期間總人口；0–1比例 | 同上；與工作年齡及高齡比例具有數學依存，不能未審查全部放入分群。 |

來源為[人口原始資料說明](../raw/taipei_population_age_raw_documentation.ipynb)、固定里界及本crosswalk。
月份存量加總只是上述比率的計算分母，不是實際累積人口總量；密度分母是每里原始幾何面積，不以全市union平均分配。
設籍人口不代表通勤、事故當事人、道路使用或交通暴露人口；都市群組比較不能推論某年齡居民個人交通事故機率。

### Supplemental Features — Feasibility Only
| 候選變數 | 可建立性／候選公式 | 尚需注意 |
| --- | --- | --- |
| `working_age_population_ratio` | 15–64歲月份人口和÷月份總人口和 | 與兒童及高齡比例合計1，三者不可直接全部納入。 |
| `oldest_old_population_ratio` | 可由逐歲欄計80歲以上或85歲以上比例 | 年齡界線尚未定案，不自選一種作正式特徵；100歲以上是頂端合併欄。 |
| `aging_index` | 65歲以上人口和÷0–14歲人口和×100 | 每百名兒童對應高齡人口，兒童分母0則NA；與主要比例有代數關係。 |
| `dependency_ratio` | （0–14歲＋65歲以上）月份人口和÷15–64歲月份人口和×100 | 每百名工作年齡人口的年齡扶養比，非實際經濟扶養；分母0則NA。 |
| `female_population_ratio` | 同月份「女」總人口和÷「計」總人口和 | 男女總計分開對齊，不把三種性別列重複加總。 |
| `population_change_rate` | 可比較2025與2024年度平均人口百分比變化；12月對12月為另一候選 | 基準年／時間窗須明列，兩者不可混用；不是出生率／淨遷徙率，分母0則NA。 |

本單元不將補充變數定為主分群輸入，也未對其執行完整EDA或模型。


In [6]:
total=research.loc[research['性別'].eq('計')].copy()
assert len(total)==10944
CHILD=[f'{age}歲數量' for age in range(15)]
ELDERLY=[f'{age}歲數量' for age in range(65,100)]+['100歲以上']
WORKING=[f'{age}歲數量' for age in range(15,65)]
total['child_count']=total[CHILD].sum(axis=1)
total['elderly_count']=total[ELDERLY].sum(axis=1)
total['working_count']=total[WORKING].sum(axis=1)
assert (total.child_count+total.elderly_count+total.working_count).eq(total['總計']).all()
joined=total.merge(crosswalk,left_on='區域代碼',right_on='population_code',how='left',validate='many_to_one',indicator=True)
assert len(joined)==len(total) and joined['_merge'].eq('both').all()
assert not joined.duplicated(['population_code','year','month']).any()
assert not set(total['區域代碼'])-set(joined.population_code)
area=pd.DataFrame({'village_id':boundary.CPTVID.astype('string'),'area_km2':boundary.geometry.area/1e6})
assert area.village_id.is_unique and np.isfinite(area.area_km2).all() and area.area_km2.gt(0).all()
def aggregate(frame,keys,expected_months):
    result=frame.groupby(keys).agg(months=('month','size'),
        denominator_population_months=('總計','sum'),elderly_population_months=('elderly_count','sum'),
        child_population_months=('child_count','sum'),working_population_months=('working_count','sum'),
        min_monthly_population=('總計','min'),max_monthly_population=('總計','max'),
        village_id=('village_id','first'),verification_status=('verification_status','first')).reset_index()
    result['period_complete']=result.months.eq(expected_months)
    result['annual_mean_population']=(result.denominator_population_months/expected_months).where(result.period_complete)
    valid=result.period_complete & result.denominator_population_months.gt(0)
    result['elderly_population_ratio']=(result.elderly_population_months/result.denominator_population_months).where(valid)
    result['child_population_ratio']=(result.child_population_months/result.denominator_population_months).where(valid)
    result=result.merge(area,on='village_id',how='left',validate='many_to_one')
    assert result.area_km2.notna().all() and result.area_km2.gt(0).all()
    result['diagnostic_density_including_provisional']=result.annual_mean_population/result.area_km2
    result['integration_allowed']=result.verification_status.isin(VERIFIED)
    result['population_density']=result.diagnostic_density_including_provisional.where(result.integration_allowed)
    return result
annual=aggregate(joined,['population_code','year'],12)
pooled=aggregate(joined,['population_code'],24)
assert len(annual)==912 and len(pooled)==456
assert not annual.duplicated(['population_code','year']).any()
assert annual.period_complete.all() and pooled.period_complete.all()
assert annual.annual_mean_population.between(annual.min_monthly_population,annual.max_monthly_population).all()
assert pooled.annual_mean_population.between(pooled.min_monthly_population,pooled.max_monthly_population).all()
for frame in [annual,pooled]:
    for variable in ['elderly_population_ratio','child_population_ratio']:
        assert frame.loc[frame[variable].notna(),variable].between(0,1).all()
    working_ratio=frame.working_population_months/frame.denominator_population_months.replace(0,np.nan)
    valid=frame.denominator_population_months.gt(0)
    np.testing.assert_allclose((working_ratio+frame.elderly_population_ratio+frame.child_population_ratio).loc[valid],1)
recombined=annual.groupby('population_code').agg(mean_from_two_complete_years=('annual_mean_population','mean'),
     elderly_numerator=('elderly_population_months','sum'),child_numerator=('child_population_months','sum'),denominator=('denominator_population_months','sum'))
direct=pooled.set_index('population_code').reindex(recombined.index)
np.testing.assert_allclose(recombined.mean_from_two_complete_years,direct.annual_mean_population,rtol=1e-12)
np.testing.assert_allclose(recombined.elderly_numerator/recombined.denominator,direct.elderly_population_ratio,rtol=1e-12)
np.testing.assert_allclose(recombined.child_numerator/recombined.denominator,direct.child_population_ratio,rtol=1e-12)
np.testing.assert_allclose(recombined.mean_from_two_complete_years/direct.area_km2,
                           direct.diagnostic_density_including_provisional,rtol=1e-12)
# Synthetic safeguards: an incomplete year or zero denominator must not become a valid annual feature.
synthetic_code=joined.population_code.iloc[0]
full_year=joined.loc[joined.population_code.eq(synthetic_code)&joined.year.eq(2024)].sort_values('month').copy()
assert len(full_year)==12
incomplete=aggregate(full_year.iloc[:11],['population_code','year'],12)
assert len(incomplete)==1 and incomplete.months.eq(11).all()
assert incomplete.annual_mean_population.isna().all() and incomplete.elderly_population_ratio.isna().all()
zero=full_year.copy()
assert len(zero)==12
zero[['總計','elderly_count','child_count','working_count']]=0
zero_result=aggregate(zero,['population_code','year'],12)
assert zero_result.period_complete.all() and zero_result.denominator_population_months.eq(0).all()
assert zero_result.elderly_population_ratio.isna().all() and zero_result.child_population_ratio.isna().all()
show('表18：年度試算實際前十列，非正式年度資料',annual[['population_code','year','village_id','verification_status','months','annual_mean_population','area_km2','population_density','elderly_population_ratio','child_population_ratio']].head(10))
descriptive=[]
for year,frame in annual.groupby('year'):
    for variable in ['annual_mean_population','population_density','elderly_population_ratio','child_population_ratio']:
        stats=frame[variable].describe(percentiles=[.25,.5,.75]).to_dict()
        descriptive.append({'year':year,'variable':variable,'missing_or_policy_pending':int(frame[variable].isna().sum()),**stats})
show('表19：必要年度品質摘要；主要密度排除暫定配對但不刪人口列',pd.DataFrame(descriptive))
show('表20：待審配對對兩年度完整性的影響',annual.loc[~annual.integration_allowed,['population_code','year','village_id','verification_status','annual_mean_population','population_density','diagnostic_density_including_provisional']])
show('表21：零人口分母或缺月／缺面積異常',annual.loc[annual.denominator_population_months.le(0)|~annual.period_complete|annual.area_km2.isna()])
show('表22：診斷密度最高及最低各三里，含配對狀態',pd.concat([annual.nlargest(3,'diagnostic_density_including_provisional'),annual.nsmallest(3,'diagnostic_density_including_provisional')])[['population_code','year','verification_status','annual_mean_population','area_km2','diagnostic_density_including_provisional']])
trial_summary={'monthly_all_sexes_rows':len(research),'monthly_total_sex_rows':len(total),'annual_rows_in_memory':len(annual),
   'pooled_rows_in_memory':len(pooled),'strict_verified_villages':int(crosswalk.verification_status.isin(VERIFIED).sum()),
   'provisional_villages':int(crosswalk.verification_status.eq('provisional_match').sum()),
   'unmatched_villages':int(crosswalk.verification_status.eq('unmatched').sum()),
   'verified_monthly_total_records':int(joined.verification_status.isin(VERIFIED).sum()),
   'provisional_monthly_total_records':int(joined.verification_status.eq('provisional_match').sum()),
   'strict_verified_annual_rows':int(annual.integration_allowed.sum()),'pending_annual_rows':int((~annual.integration_allowed).sum()),
   'annual_zero_population_denominators':int(annual.denominator_population_months.eq(0).sum()),
   'annual_incomplete_groups':int((~annual.period_complete).sum()),'code_sets_same_in_2024_2025':True,
   'official_snapshots_taipei_codes_names_identical':True,'local_raw_name_changes':0,
   'mean_and_ratios_recombined_equal_direct_24_months':True,'missing_month_and_zero_denominator_tests_passed':True,
   'fixed_geometry_areas_positive':True,'join_preserves_every_population_row':True,
   'formal_annual_population_exports':0,'accident_joins':0}
print(trial_summary)


**表18：年度試算實際前十列，非正式年度資料**

,population_code,year,village_id,verification_status,months,annual_mean_population,area_km2,population_density,elderly_population_ratio,child_population_ratio
0,63000010002,2024,6300100002,verified_exact_match,12,5018.916667,1.281723,3915.757309,0.237236,0.112674
1,63000010002,2025,6300100002,verified_exact_match,12,4884.583333,1.281723,3810.950482,0.243231,0.112514
2,63000010003,2024,6300100003,verified_exact_match,12,8039.083333,0.259347,30997.384837,0.244576,0.168904
3,63000010003,2025,6300100003,verified_exact_match,12,7874.416667,0.259347,30362.457218,0.250749,0.167314
4,63000010004,2024,6300100004,verified_exact_match,12,6591.333333,0.134542,48990.793465,0.265892,0.111535
5,63000010004,2025,6300100004,verified_exact_match,12,6440.916667,0.134542,47872.805422,0.273331,0.105135
6,63000010005,2024,6300100005,verified_exact_match,12,4387.000000,0.115138,38102.243148,0.257370,0.096801
7,63000010005,2025,6300100005,verified_exact_match,12,4311.500000,0.115138,37446.505888,0.266922,0.097974
8,63000010006,2024,6300100006,verified_exact_match,12,5074.500000,0.099809,50841.924014,0.257612,0.129093
9,63000010006,2025,6300100006,verified_exact_match,12,4943.083333,0.099809,49525.247262,0.267680,0.125529


**表19：必要年度品質摘要；主要密度排除暫定配對但不刪人口列**

,year,variable,missing_or_policy_pending,count,mean,std,min,25%,50%,75%,max
0,2024,annual_mean_population,0,456.0,5489.632493,1976.909218,856.250000,4199.520833,5316.041667,6717.562500,12090.333333
1,2024,population_density,1,455.0,27561.947266,18131.571216,52.600089,13447.999921,27077.844668,38990.601674,107146.447060
2,2024,elderly_population_ratio,0,456.0,0.229860,0.034261,0.132529,0.208489,0.230985,0.251993,0.334525
3,2024,child_population_ratio,0,456.0,0.118993,0.037680,0.057276,0.091334,0.109768,0.137924,0.267030
4,2025,annual_mean_population,0,456.0,5386.794408,1943.437693,749.833333,4134.666667,5173.583333,6601.645833,11729.750000
5,2025,population_density,1,455.0,27031.585673,17735.398034,51.655697,13119.151386,26540.342506,38332.669027,104127.497798
6,2025,elderly_population_ratio,0,456.0,0.240855,0.034549,0.140793,0.218539,0.242536,0.263181,0.347787
7,2025,child_population_ratio,0,456.0,0.117489,0.037853,0.055211,0.090021,0.108476,0.136774,0.268630


**表20：待審配對對兩年度完整性的影響**

,population_code,year,village_id,verification_status,annual_mean_population,population_density,diagnostic_density_including_provisional
478,63000070015,2024,6300700015,provisional_match,5064.750000,NaN,33436.682753
479,63000070015,2025,6300700015,provisional_match,4916.583333,NaN,32458.509728


**表21：零人口分母或缺月／缺面積異常**

無符合紀錄。

**表22：診斷密度最高及最低各三里，含配對狀態**

,population_code,year,verification_status,annual_mean_population,area_km2,diagnostic_density_including_provisional
518,63000070035,2024,verified_exact_match,3401.250000,0.031744,107146.447060
519,63000070035,2025,verified_exact_match,3305.416667,0.031744,104127.497798
216,63000030037,2024,verified_exact_match,6735.583333,0.068833,97853.558888
911,63000120042,2025,verified_exact_match,843.250000,16.324434,51.655697
910,63000120042,2024,verified_exact_match,858.666667,16.324434,52.600089
819,63000110047,2025,verified_exact_match,1448.750000,11.720427,123.608973


{'monthly_all_sexes_rows': 32832, 'monthly_total_sex_rows': 10944, 'annual_rows_in_memory': 912, 'pooled_rows_in_memory': 456, 'strict_verified_villages': 455, 'provisional_villages': 1, 'unmatched_villages': 0, 'verified_monthly_total_records': 10920, 'provisional_monthly_total_records': 24, 'strict_verified_annual_rows': 910, 'pending_annual_rows': 2, 'annual_zero_population_denominators': 0, 'annual_incomplete_groups': 0, 'code_sets_same_in_2024_2025': True, 'official_snapshots_taipei_codes_names_identical': True, 'local_raw_name_changes': 0, 'mean_and_ratios_recombined_equal_direct_24_months': True, 'missing_month_and_zero_denominator_tests_passed': True, 'fixed_geometry_areas_positive': True, 'join_preserves_every_population_row': True, 'formal_annual_population_exports': 0, 'accident_joins': 0}


## 6. Save the Authorized Crosswalk and Reopen It
只輸出crosswalk，UTF-8-SIG、LF、固定欄序及village_id排序；所有識別欄以字串讀回。
若檔案已存在，只接受與本次確定性產物bytes完全相同，否則停止，避免覆寫不同版本。


In [7]:
output=file('data/processed/village_crosswalk.csv',area='data/processed',must_exist=False)
assert output not in [file(p) for p in EXPECTED_SHA256]
payload=crosswalk.to_csv(index=False,lineterminator='\n').encode('utf-8-sig')
if output.exists():
    assert output.read_bytes()==payload,'Existing crosswalk differs; stop and review version before replacement'
else:
    with output.open('xb') as stream:stream.write(payload)
saved=pd.read_csv(output,encoding='utf-8-sig',dtype='string',keep_default_na=False)
pd.testing.assert_frame_equal(saved,crosswalk)
assert list(saved.columns)==COLUMNS and len(saved)==456
assert saved[['village_id','population_code','boundary_code']].nunique().eq(456).all()
assert saved.population_code.str.len().eq(11).all() and saved.boundary_code.str.len().eq(10).all()
assert saved.loc[saved.population_code.eq('63000070015'),'verification_status'].iloc[0]=='provisional_match'
output_manifest={'path':'data/processed/village_crosswalk.csv','role':'crosswalk','rows':len(saved),'columns':list(saved.columns),
    'sha256':sha(output),'bytes':output.stat().st_size,'verification':'passed serialized read-back; provisional status retained',
    'status_counts':saved.verification_status.value_counts().to_dict(),
    'processing_notebook':'data/code/taipei_population_validation.ipynb',
    'documentation':'data/processed/village_crosswalk_documentation.ipynb'}
show('表23：實際磁碟crosswalk與hash',pd.DataFrame([output_manifest]))
print('SHA256',output_manifest['sha256'])


**表23：實際磁碟crosswalk與hash**

,path,role,rows,columns,sha256,bytes,verification,status_counts,processing_notebook,documentation
0,data/processed/village_crosswalk.csv,crosswalk,456,"[village_id, population_code, boundary_code, district_name, population_village_name, boundary_village_name...",dd9e762419c302cf5b4bb19daad2673d7d54d0bb750204e7e33ecbf1a9983f69,362330,passed serialized read-back; provisional status retained,"{'verified_exact_match': 455, 'provisional_match': 1}",data/code/taipei_population_validation.ipynb,data/processed/village_crosswalk_documentation.ipynb


SHA256 dd9e762419c302cf5b4bb19daad2673d7d54d0bb750204e7e33ecbf1a9983f69


## 7. Quality Conclusions and Sequential Handoff
人口24個月完整性及計算規則經驗證；crosswalk以456列保留全部里，其中455精確名稱配對、1高可信暫定配對。
官方五快照支持糖廍里的11碼及正式名稱，未取得CPTVID官方轉換或名稱誤植更正文件，所以不升級verified_official。
正式整合必須顯式檢查狀態；暫定里仍保留人口，診斷試算與可用主要密度分開，不能默默刪里或把912列全稱已完成地理整合。
已接受固定456里、EPSG:3826幾何面積作密度分母；歷史空間錯配、孔洞／重疊及事故datum未知限制保留。
官方人口來源只說戶籍登記產製，精確月內統計基準日／年齡計算細則仍未補證；不將設籍人口視作交通暴露。
本次只建立crosswalk，不輸出年度人口、事故歸里、都市計畫疊合或分群。完成後停止，等待審查，不進入Unit 1.4。


In [8]:
assert boundary.geometry.to_wkb().equals(original_geometry)
verify(EXPECTED_SHA256)
assert sha(file('data/processed/village_crosswalk.csv'))==output_manifest['sha256']
temp=ROOT/'data/temp/unit13_validation';temp.mkdir(parents=True,exist_ok=True)
(temp/'trial_summary.json').write_text(json.dumps(trial_summary,ensure_ascii=False,indent=2),encoding='utf-8')
(temp/'crosswalk_manifest.json').write_text(json.dumps(output_manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print({'all_bound_inputs_unchanged':True,'raw_geometry_unchanged':True,'formal_derived_tables':['data/processed/village_crosswalk.csv'],
       'annual_population_files_written':0,'old_notebooks_executed':0,'accident_CRS':'unconfirmed','next_unit_started':False})


{'all_bound_inputs_unchanged': True, 'raw_geometry_unchanged': True, 'formal_derived_tables': ['data/processed/village_crosswalk.csv'], 'annual_population_files_written': 0, 'old_notebooks_executed': 0, 'accident_CRS': 'unconfirmed', 'next_unit_started': False}
